# dataloader-batching — worked example 3: Shuffle reorders batches across epochs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-batching`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

With `shuffle=True`, a `DataLoader` draws a fresh random permutation of the dataset at the start of every epoch, so the sequence of examples differs from one pass to the next. With `shuffle=False`, every epoch yields the identical order — which is what you want for reproducible evaluation. The set of examples is the same either way; only the ordering changes.

## Worked solution

**Goal:** demonstrate that a shuffled loader produces different epoch orderings while an unshuffled one repeats.

1. Re-seed with `t.manual_seed(0)` and build a dataset of unique ids `t.arange(N)` so the flattened epoch order is directly readable.
2. Build one shuffled loader. To capture an epoch's full ordering, iterate it and `t.cat` all batch id-tensors into a single vector — that vector *is* the visitation order.
3. Run TWO epochs over the shuffled loader (just iterate it twice). PyTorch re-permutes per epoch, so the two flattened orderings should differ. We compare them with `t.equal`; expecting `False`.
4. For contrast, build a second loader with `shuffle=False` and capture two epochs the same way; these orderings must be identical (`t.equal` -> `True`).
5. We print both booleans. The key idea: capture *order*, not just contents — so we keep the concatenated sequence rather than sorting it.

**Why it works:** the loader's sampler is a `RandomSampler` when `shuffle=True`; it samples a new permutation each time `iter()` is called, which happens once per `for` loop. `shuffle=False` uses a `SequentialSampler`, which is deterministic.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def epoch_order(loader):
    return t.cat([ids for (ids,) in loader])


def compare_orderings(N, batch_size):
    t.manual_seed(0)
    ds = TensorDataset(t.arange(N))
    shuf = DataLoader(ds, batch_size=batch_size, shuffle=True)
    noshuf = DataLoader(ds, batch_size=batch_size, shuffle=False)
    shuf_same = t.equal(epoch_order(shuf), epoch_order(shuf))
    noshuf_same = t.equal(epoch_order(noshuf), epoch_order(noshuf))
    return shuf_same, noshuf_same


shuf_same, noshuf_same = compare_orderings(40, 7)
print('shuffle=True epochs identical:', shuf_same)
print('shuffle=False epochs identical:', noshuf_same)